<a href="https://colab.research.google.com/github/tanushhh21/tanush21/blob/main/notebook_condition_E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets torch accelerate zstandard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 96.6 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

In [ ]:
import torch, numpy as np, json, re, warnings, zlib, matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.stats import pointbiserialr, mannwhitneyu, ttest_ind
from scipy.interpolate import interp1d
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
torch.manual_seed(42)
np.random.seed(42)

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
N_EVAL = 60
N_CONTRASTIVE = 80
MAX_NEW_TOKENS = 400

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16,
    trust_remote_code=True
).to(DEVICE)
model.eval()
print(f'Model loaded. Layers: {model.config.num_hidden_layers}')

Loading Qwen/Qwen2.5-1.5B-Instruct...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded. Layers: 28


In [ ]:
print("Model device:", next(model.parameters()).device)

Model device: cuda:0


In [ ]:
gsm8k_test  = load_dataset('openai/gsm8k', 'main', split='test')
gsm8k_train = load_dataset('openai/gsm8k', 'main', split='train')

eval_problems  = [{'question': ex['question'], 'answer': ex['answer']} for ex in gsm8k_test.select(range(N_EVAL))]
train_problems = [{'question': ex['question'], 'answer': ex['answer']} for ex in gsm8k_train.select(range(N_CONTRASTIVE))]

print(f'Eval: {len(eval_problems)} | Train (for steering): {len(train_problems)}')

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Eval: 60 | Train (for steering): 80


In [ ]:
# ── Answer extraction ──────────────────────────────────────────────────────────
def extract_gold_answer(s):
    m = re.search(r'####\s*([\d,.-]+)', s)
    return m.group(1).replace(',','').strip() if m else None

def extract_predicted_answer(text):
    nums = re.findall(r'-?[\d,]+(?:\.\d+)?', text.replace(',',''))
    return nums[-1].replace(',','') if nums else None

def is_correct(pred_text, gold_str):
    gold = extract_gold_answer(gold_str)
    pred = extract_predicted_answer(pred_text)
    if not gold or not pred: return False
    try:    return abs(float(pred) - float(gold)) < 1e-6
    except: return pred.strip() == gold.strip()

# ── Prompts ────────────────────────────────────────────────────────────────────
SYSTEM = 'You are a helpful math tutor. Solve the problem step by step.'

def make_prompt(question, hint=None):
    content = question if not hint else f'{question}\n\nHint: {hint}'
    msgs = [{'role':'system','content':SYSTEM}, {'role':'user','content':content}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def make_cot_prompt(question, cot):
    msgs = [{'role':'system','content':SYSTEM},{'role':'user','content':question},{'role':'assistant','content':cot}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

def make_direct_prompt(question, answer):
    msgs = [{'role':'system','content':SYSTEM},{'role':'user','content':question},{'role':'assistant','content':answer}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

def tokenize(text, max_length=1024):
    return tokenizer(text, return_tensors='pt', truncation=True, max_length=max_length).input_ids.to(DEVICE)

print('Utilities ready.')

Utilities ready.


In [ ]:
def compression_density(text):
    if not text: return 0.0
    raw  = len(text.encode('utf-8'))
    comp = len(zlib.compress(text.encode('utf-8')))
    return 1.0 - (comp / raw)

def parse_steps(text):
    for pat in [r'(?=Step\s*\d+[:\.])', r'(?=\n\d+[\.)\s])', r'(?=\n\n)']:
        parts = [p.strip() for p in re.split(pat, text, flags=re.IGNORECASE) if p.strip() and len(p.strip())>10]
        if len(parts) >= 3: return parts
    parts = [p.strip() for p in text.split('\n') if p.strip() and len(p.strip())>10]
    return parts if len(parts) >= 2 else [text]

def id_variance(density_result):
    """
    Variance of min-max normalised LP scores.
    FIX: do NOT z-score before computing variance — z-scoring forces var=1.0 by definition.
    Min-max scales to [0,1] while preserving the actual spread across steps.
    """
    if density_result is None or len(density_result['lp_scores']) < 2:
        return None
    lp  = np.array(density_result['lp_scores'])
    rng = lp.max() - lp.min()
    if rng < 1e-9: return 0.0
    return float(np.var((lp - lp.min()) / rng))

@torch.no_grad()
def compute_step_density(model, tokenizer, prompt_text, response_text):
    steps = parse_steps(response_text)
    if len(steps) < 2: return None

    full_text = prompt_text + response_text
    enc = tokenizer(full_text, return_tensors='pt', truncation=True, max_length=2048).to(DEVICE)
    input_ids = enc.input_ids
    logits    = model(input_ids, return_dict=True).logits[0]
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
    probs     = torch.exp(log_probs)

    prompt_len = min(
        len(tokenizer(prompt_text, truncation=True, max_length=2048).input_ids),
        input_ids.shape[1] - 1
    )
    resp_tok_texts = [tokenizer.decode([t]) for t in input_ids[0, prompt_len:]]

    token_cursor = 0
    step_ranges  = []
    for step_text in steps:
        start, chars = token_cursor, 0
        while token_cursor < len(resp_tok_texts) and chars < len(step_text):
            chars += len(resp_tok_texts[token_cursor]); token_cursor += 1
        step_ranges.append((start, max(token_cursor, start+1)))

    lp_scores, h_scores, d_scores = [], [], []
    for (start, end), step_text in zip(step_ranges, steps):
        end     = min(end, len(resp_tok_texts))
        g_start = prompt_len + start
        g_end   = min(prompt_len + end, logits.shape[0])
        p_start = max(0, g_start - 1)
        p_end   = g_end - 1
        if p_start >= p_end or p_end > log_probs.shape[0]: continue
        actual  = input_ids[0, g_start:g_end]
        n       = min(len(actual), p_end - p_start)
        if n < 1: continue
        tlp = log_probs[p_start:p_start+n]
        alp = tlp[torch.arange(n), actual[:n]]
        lp_scores.append(-alp.mean().item())
        h_scores.append(-(probs[p_start:p_start+n] * log_probs[p_start:p_start+n]).sum(-1).mean().item())
        d_scores.append(compression_density(step_text))

    if len(lp_scores) < 2: return None
    return {'lp_scores': lp_scores, 'h_scores': h_scores, 'd_scores': d_scores, 'n_steps': len(lp_scores)}

print('Density functions ready.')

Density functions ready.


In [ ]:
@torch.no_grad()
def generate_greedy(prompt_text, max_new_tokens=MAX_NEW_TOKENS):
    ids = tokenize(prompt_text)
    out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

@torch.no_grad()
def generate_with_steered_cache(prompt_text, steering_vectors, ck=0.0, cv=4.0,
                                 max_new_tokens=MAX_NEW_TOKENS):
    input_ids = tokenize(prompt_text)
    seq_len   = input_ids.shape[1]
    target_pos = seq_len - 1

    outputs  = model(input_ids, use_cache=True, return_dict=True)
    kv_cache = outputs.past_key_values

    # Modify KV cache — handles both DynamicCache and legacy tuple formats
    if hasattr(kv_cache, 'key_cache'):
        for li in range(min(len(kv_cache.key_cache), len(steering_vectors['k']))):
            sk = steering_vectors['k'][li].to(kv_cache.key_cache[li].device)
            sv = steering_vectors['v'][li].to(kv_cache.value_cache[li].device)
            kv_cache.key_cache[li][:, :, target_pos, :]   += ck * sk
            kv_cache.value_cache[li][:, :, target_pos, :] += cv * sv
    else:
        modified = []
        for li, (k, v) in enumerate(kv_cache):
            k_mod, v_mod = k.clone(), v.clone()
            if li < len(steering_vectors['k']):
                k_mod[:, :, target_pos, :] += ck * steering_vectors['k'][li].to(k.device)
                v_mod[:, :, target_pos, :] += cv * steering_vectors['v'][li].to(v.device)
            modified.append((k_mod, v_mod))
        kv_cache = tuple(modified)

    # Autoregressive decoding from modified cache
    current_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    generated = [current_token.item()]
    past_kv   = kv_cache
    for _ in range(max_new_tokens - 1):
        out = model(current_token, past_key_values=past_kv, use_cache=True, return_dict=True)
        current_token = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        past_kv = out.past_key_values
        tok = current_token.item()
        generated.append(tok)
        if tok == tokenizer.eos_token_id: break

    return tokenizer.decode(generated, skip_special_tokens=True)

print('Generation functions ready.')

Generation functions ready.


In [ ]:
HINT_S1 = 'Question: {q}\n\nI was told the answer is {a} but I do not know why. Please explain step by step.'
HINT_S2_SYS = ('You are a tutor. Given a question, answer and solution, write a concise '
               'high-level hint (1-2 sentences). Do NOT reveal calculations or the final answer.')
HINT_S2 = 'Question: {q}\nAnswer: {a}\nSolution: {s}\n\nProvide an abstract hint:'

def generate_abstract_hint(question, gold_answer_str):
    gold = extract_gold_answer(gold_answer_str)
    # Stage 1
    p1 = tokenizer.apply_chat_template(
        [{'role':'system','content':'You are a math tutor.'},
         {'role':'user','content':HINT_S1.format(q=question, a=gold)}],
        tokenize=False, add_generation_prompt=True)
    explanation = generate_greedy(p1, max_new_tokens=200)
    # Stage 2
    p2 = tokenizer.apply_chat_template(
        [{'role':'system','content':HINT_S2_SYS},
         {'role':'user','content':HINT_S2.format(q=question, a=gold, s=explanation[:500])}],
        tokenize=False, add_generation_prompt=True)
    hint = generate_greedy(p2, max_new_tokens=80)
    if gold and gold in hint:
        hint = hint.replace(gold, '[value]')
    return hint.strip()

print('Generating abstract hints for all eval problems...')
hints = []
for i, prob in enumerate(tqdm(eval_problems)):
    hints.append(generate_abstract_hint(prob['question'], prob['answer']))
    if i < 3:
        print(f'\n[{i}] Q: {prob["question"][:70]}...')
        print(f'     Hint: {hints[-1]}')
print(f'\nGenerated {len(hints)} hints.')

Generating abstract hints for all eval problems...


  0%|          | 0/60 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



[0] Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every ...
     Hint: Janet's earnings come from selling leftover eggs. Calculate remaining eggs and multiply by price.

[1] Q: A robe takes 2 bolts of blue fiber and half that much white fiber.  Ho...
     Hint: Calculate the total bolts based on the ratio provided.

[2] Q: Josh decides to try flipping a house.  He buys a house for $80,000 and...
     Hint: Calculate the net worth and assess the profit from the renovation project.

Generated 60 hints.


In [ ]:
@torch.no_grad()
def get_last_token_kv(text, max_length=512):
    ids = tokenizer(text, return_tensors='pt', truncation=True, max_length=max_length).input_ids.to(DEVICE)
    kv  = model(ids, use_cache=True, return_dict=True).past_key_values
    keys, vals = [], []
    for layer in kv.layers:
        keys.append(layer.keys[0, :, -1, :].float().cpu())
        vals.append(layer.values[0, :, -1, :].float().cpu())
    return keys, vals


@torch.no_grad()
def generate_with_steered_cache(prompt_text, steering_vectors, ck=0.0, cv=1.0,
                                 max_new_tokens=MAX_NEW_TOKENS):
    input_ids = tokenize(prompt_text)
    target_pos = input_ids.shape[1] - 1

    outputs  = model(input_ids, use_cache=True, return_dict=True)
    kv_cache = outputs.past_key_values

    # Modify KV cache at last prompt token position
    for li, layer in enumerate(kv_cache.layers):
        if li < len(steering_vectors['k']):
            layer.keys[0, :, target_pos, :]   += ck * steering_vectors['k'][li].to(layer.keys.device)
            layer.values[0, :, target_pos, :] += cv * steering_vectors['v'][li].to(layer.values.device)

    # Autoregressive decoding
    current_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    generated = [current_token.item()]
    past_kv   = kv_cache
    eos_count = 0
    for _ in range(max_new_tokens - 1):
        out = model(current_token, past_key_values=past_kv, use_cache=True, return_dict=True)
        current_token = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        past_kv = out.past_key_values
        tok = current_token.item()
        generated.append(tok)
        if tok == tokenizer.eos_token_id:
            eos_count += 1
            if eos_count >= 2 or len(generated) > 10:
                break

    return tokenizer.decode(generated, skip_special_tokens=True)

In [ ]:
# Rebuild steering vectors from scratch
print("Rebuilding steering vectors...")

sum_k, sum_v, n_valid = None, None, 0

for prob in tqdm(train_problems[:20]):  # use only 20 for speed
    question = prob['question']
    gold     = extract_gold_answer(prob['answer'])
    if gold is None:
        continue
    try:
        cot      = generate_greedy(make_prompt(question), max_new_tokens=150)
        pos_text = make_cot_prompt(question, cot)
        neg_text = make_direct_prompt(question, f'The answer is {gold}.')
        k_pos, v_pos = get_last_token_kv(pos_text)
        k_neg, v_neg = get_last_token_kv(neg_text)
    except Exception as e:
        print(f"  Skipping: {e}")
        continue
    if len(k_pos) != len(k_neg):
        continue
    kd = [kp - kn for kp, kn in zip(k_pos, k_neg)]
    vd = [vp - vn for vp, vn in zip(v_pos, v_neg)]
    if sum_k is None:
        sum_k, sum_v = kd, vd
    else:
        sum_k = [s+d for s,d in zip(sum_k, kd)]
        sum_v = [s+d for s,d in zip(sum_v, vd)]
    n_valid += 1
    print(f"  Pair {n_valid} done")

if n_valid == 0:
    print("ERROR: still no valid pairs. Check get_last_token_kv function.")
else:
    steering_vectors = {
        'k': [s/n_valid for s in sum_k],
        'v': [s/n_valid for s in sum_v]
    }
    print(f"Done. {n_valid} pairs. Layers: {len(steering_vectors['k'])}")
    print(f"Shape: {steering_vectors['k'][0].shape}")

Rebuilding steering vectors...


  0%|          | 0/20 [00:00<?, ?it/s]

  Pair 1 done
  Pair 2 done
  Pair 3 done
  Pair 4 done
  Pair 5 done
  Pair 6 done
  Pair 7 done
  Pair 8 done
  Pair 9 done
  Pair 10 done
  Pair 11 done
  Pair 12 done
  Pair 13 done
  Pair 14 done
  Pair 15 done
  Pair 16 done
  Pair 17 done
  Pair 18 done
  Pair 19 done
  Pair 20 done
Done. 20 pairs. Layers: 28
Shape: torch.Size([2, 128])


In [ ]:
# ── Conditions A, B, C, D ────────────────────────────────────────────────────
# run_condition helper
def run_condition(name, prompt_fn, desc=""):
    print(f"\n{'='*55}\nCondition {name}: {desc}\n{'='*55}")
    results = []
    for i, prob in enumerate(tqdm(eval_problems)):
        q, ans, hint = prob["question"], prob["answer"], hints[i]
        try:
            response = prompt_fn(q, hint)
        except Exception as ex:
            print(f"  Problem {i}: {ex}")
            response = ""
        correct  = is_correct(response, ans)
        prompt   = make_prompt(q)
        density  = compute_step_density(model, tokenizer, prompt, response)
        variance = id_variance(density)
        steps    = parse_steps(response)
        results.append({
            "idx": i, "correct": correct, "variance": variance,
            "density": density, "response": response,
            "n_tokens": len(tokenizer(response)["input_ids"]),
            "n_steps":  len(steps),
        })
        if (i+1) % 20 == 0:
            acc = sum(r["correct"] for r in results)/len(results)*100
            print(f"  [{i+1}/60] Accuracy: {acc:.1f}%")
    acc = sum(r["correct"] for r in results)/len(results)*100
    print(f"  Final accuracy: {acc:.1f}%")
    return results

CK, CV = 0.0, 1.0

results_A = run_condition("A", lambda q,h: generate_greedy(make_prompt(q)),
                           desc="Baseline: greedy CoT")
results_B = run_condition("B", lambda q,h: generate_greedy(make_prompt(q, hint=h)),
                           desc="Hint: NuRL abstract hint")
results_C = run_condition("C", lambda q,h: generate_with_steered_cache(make_prompt(q), steering_vectors, ck=CK, cv=CV),
                           desc="Cache Steer")
results_D = run_condition("D", lambda q,h: generate_with_steered_cache(make_prompt(q, hint=h), steering_vectors, ck=CK, cv=CV),
                           desc="Combined: hint + cache steer")

all_conditions = {
    "A: Baseline":    results_A,
    "B: Hint":        results_B,
    "C: Cache Steer": results_C,
    "D: Combined":    results_D,
}

import numpy as np
print("\n" + "="*65)
print(f"{'Condition':<22} {'Acc':>7} {'Var(C)':>8} {'Var(W)':>8} {'Gap':>8}")
print("-"*55)
summary = {}
for cname, results in all_conditions.items():
    vc  = [r["variance"] for r in results if r["correct"]     and r["variance"] is not None]
    vw  = [r["variance"] for r in results if not r["correct"] and r["variance"] is not None]
    acc = np.mean([r["correct"] for r in results])*100
    mvc = float(np.mean(vc)) if vc else float("nan")
    mvw = float(np.mean(vw)) if vw else float("nan")
    summary[cname] = {"accuracy": acc, "var_correct": mvc, "var_wrong": mvw, "vars_c": vc, "vars_w": vw}
    print(f"{cname:<22} {acc:>7.1f}% {mvc:>8.4f} {mvw:>8.4f} {mvc-mvw:>+8.4f}")



Condition A: Baseline: greedy CoT


  0%|          | 0/60 [00:00<?, ?it/s]

  [40/60] Accuracy: 50.0%
  [60/60] Accuracy: 51.7%
  Final accuracy: 51.7%

Condition B: Hint: NuRL abstract hint


  0%|          | 0/60 [00:00<?, ?it/s]

  [20/60] Accuracy: 45.0%
  [40/60] Accuracy: 42.5%
  [60/60] Accuracy: 46.7%
  Final accuracy: 46.7%

Condition C: Cache Steer


  0%|          | 0/60 [00:00<?, ?it/s]

  [20/60] Accuracy: 50.0%
  [40/60] Accuracy: 52.5%
  [60/60] Accuracy: 53.3%
  Final accuracy: 53.3%

Condition D: Combined: hint + cache steer


  0%|          | 0/60 [00:00<?, ?it/s]

  [20/60] Accuracy: 60.0%
  [40/60] Accuracy: 65.0%
  [60/60] Accuracy: 65.0%
  Final accuracy: 65.0%

Condition                  Acc   Var(C)   Var(W)      Gap
-------------------------------------------------------
A: Baseline               51.7%   0.1421   0.1402  +0.0018
B: Hint                   46.7%   0.1467   0.1302  +0.0165
C: Cache Steer            53.3%   0.1508   0.1470  +0.0038
D: Combined               65.0%   0.1411   0.1314  +0.0097


In [ ]:
# ── EGDR Setup: spaCy + thresholds + density direction ───────────────────────
import spacy, numpy as np, torch

try:
    nlp_cav = spacy.load("en_core_web_sm")
except:
    import subprocess
    subprocess.run(["pip","install","spacy","-q","--break-system-packages"])
    subprocess.run(["python","-m","spacy","download","en_core_web_sm"])
    nlp_cav = spacy.load("en_core_web_sm")

def caveman_ratio(text):
    if not text or len(text.strip()) < 5: return 0.42
    doc = nlp_cav(text)
    total, removed = 0, 0
    for tok in doc:
        if tok.is_space or tok.is_punct: continue
        total += 1
        if tok.is_stop or tok.pos_ in ("AUX","DET") or            (tok.pos_=="CCONJ" and tok.text.lower() in {"and","or"}):
            removed += 1
    return removed/total if total > 0 else 0.42

# Entropy thresholds from existing LP scores
all_lp = []
for cname, results in all_conditions.items():
    for r in results:
        if r["correct"] and r["density"] and r["density"]["lp_scores"]:
            all_lp.extend(r["density"]["lp_scores"])

H_MEAN = float(np.mean(all_lp))
H_HIGH = H_MEAN * 1.5
H_LOW  = H_MEAN * 0.6

# Density direction: normalised steering vectors
N_LAYERS = len(steering_vectors["k"])
density_direction_k = [steering_vectors["k"][li].to(DEVICE) / (steering_vectors["k"][li].norm()+1e-8) for li in range(N_LAYERS)]
density_direction_v = [steering_vectors["v"][li].to(DEVICE) / (steering_vectors["v"][li].norm()+1e-8) for li in range(N_LAYERS)]

MAX_EGDR_TOKENS = 400
print(f"EGDR setup done.")
print(f"  H_MEAN={H_MEAN:.4f} | H_HIGH={H_HIGH:.4f} | H_LOW={H_LOW:.4f}")
print(f"  Layers={N_LAYERS} | spaCy ready")


EGDR setup done.
  H_MEAN=0.1696 | H_HIGH=0.2544 | H_LOW=0.1018
  Layers=28 | spaCy ready


In [ ]:
# ── EGDR generation function ─────────────────────────────────────────────────

@torch.no_grad()
def generate_egdr(prompt_text, coeff=0.15):
    """
    EGDR: token-by-token, entropy-gated KV correction.
    Uses kv_cache.layers[li].keys/.values (transformers 5.10.2 DynamicCache).
    coeff: correction strength (positive=dense, negative*0.3=sparse).
    """
    enc       = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=512)
    input_ids = enc.input_ids.to(DEVICE)
    kv_cache  = None
    generated = []
    lp_log, ent_log = [], []

    for _ in range(MAX_EGDR_TOKENS):
        out = model(input_ids[:,-1:] if kv_cache else input_ids,
                    past_key_values=kv_cache, use_cache=True, return_dict=True)
        logits   = out.logits[0,-1,:]
        kv_cache = out.past_key_values

        log_probs = torch.log_softmax(logits.float(), dim=-1)
        probs     = torch.exp(log_probs)
        entropy   = float(-(probs*log_probs).sum().item())
        ent_log.append(entropy)

        next_tok  = logits.argmax().unsqueeze(0).unsqueeze(0)
        tok_id    = next_tok.item()
        lp_log.append(-float(log_probs[tok_id].item()))
        generated.append(tok_id)
        input_ids = torch.cat([input_ids, next_tok], dim=1)

        # Entropy gate
        direction = +coeff if entropy > H_HIGH else (-coeff*0.3 if entropy < H_LOW else 0.0)

        # KV correction via .layers (DynamicCache 5.10.2)
        if direction != 0.0:
            for li in range(min(len(kv_cache.layers), N_LAYERS)):
                kv_cache.layers[li].keys[:,:,-1,:]   += direction * density_direction_k[li]
                kv_cache.layers[li].values[:,:,-1,:] += direction * density_direction_v[li]

        if tok_id == tokenizer.eos_token_id: break
        if len(generated) > 10 and "####" in tokenizer.decode(generated, skip_special_tokens=True): break

    return tokenizer.decode(generated, skip_special_tokens=True), lp_log, ent_log

print("generate_egdr() ready. Signature: generate_egdr(prompt_text, coeff=0.15)")


generate_egdr() ready. Signature: generate_egdr(prompt_text, coeff=0.15)


In [ ]:
# ── EGDR run helper ──────────────────────────────────────────────────────────
import numpy as np

def run_egdr_condition(coeff, use_hint=False, label=""):
    """Run EGDR on all 60 problems. Returns results list."""
    print(f"\n{'='*55}\n{label} (coeff={coeff}, hint={use_hint})\n{'='*55}")
    results = []
    for i, prob in enumerate(tqdm(eval_problems)):
        q, ans, hint = prob["question"], prob["answer"], hints[i]
        prompt = make_prompt(q, hint=hint) if use_hint else make_prompt(q)

        try:
            response, lp_log, ent_log = generate_egdr(prompt, coeff=coeff)
            egdr_ok = True
        except Exception as ex:
            print(f"  Problem {i}: {ex} -- greedy fallback")
            response, lp_log, ent_log = generate_greedy(prompt), [], []
            egdr_ok = False

        correct = is_correct(response, ans)
        steps   = parse_steps(response)

        # Variance from lp_log
        if lp_log and len(steps) >= 2 and egdr_ok:
            tps      = max(1, len(lp_log)//len(steps))
            step_lps = [float(np.mean(lp_log[s*tps:(s+1)*tps if s<len(steps)-1 else len(lp_log)]))
                        for s in range(len(steps))]
            arr      = np.array(step_lps)
            rng      = arr.max()-arr.min()
            variance = float(np.var((arr-arr.min())/rng)) if rng > 1e-9 else 0.0
        else:
            variance = None

        cav = [caveman_ratio(s) for s in steps]
        results.append({
            "idx": i, "correct": correct, "variance": variance,
            "mean_caveman": float(np.mean(cav)) if cav else None,
            "mean_entropy": float(np.mean(ent_log)) if ent_log else None,
            "egdr_ok": egdr_ok,
        })

    acc  = sum(r["correct"] for r in results)/len(results)*100
    n_ok = sum(r["egdr_ok"] for r in results)
    vc   = [r["variance"]     for r in results if r["correct"]     and r["variance"] is not None]
    vw   = [r["variance"]     for r in results if not r["correct"] and r["variance"] is not None]
    cc   = [r["mean_caveman"] for r in results if r["correct"]     and r["mean_caveman"] is not None]
    cw   = [r["mean_caveman"] for r in results if not r["correct"] and r["mean_caveman"] is not None]
    mvc  = float(np.mean(vc)) if vc else float("nan")
    mvw  = float(np.mean(vw)) if vw else float("nan")
    mcc  = float(np.mean(cc)) if cc else float("nan")
    mcw  = float(np.mean(cw)) if cw else float("nan")
    print(f"  Acc={acc:.1f}% | EGDR worked={n_ok}/60")
    print(f"  Var(C)={mvc:.4f} Var(W)={mvw:.4f} LP gap={mvc-mvw:+.4f}")
    print(f"  Cav(C)={mcc:.4f} Cav(W)={mcw:.4f} Cav gap={mcc-mcw:+.4f}")
    return results, acc

print("run_egdr_condition() ready.")


run_egdr_condition() ready.


In [ ]:
# ── GRID SEARCH: coeff in {0.05, 0.10, 0.15, 0.20, 0.30} ────────────────────
# ~10-12 min per coeff value, ~1 hour total

GRID_COEFFS  = [0.05, 0.10, 0.15, 0.20, 0.30]
grid_results = {}

for coeff in GRID_COEFFS:
    res, acc = run_egdr_condition(coeff, use_hint=False, label=f"EGDR coeff={coeff}")
    grid_results[coeff] = {"results": res, "accuracy": acc}

print("\n" + "="*45)
print("GRID SEARCH SUMMARY:")
print(f"{'Coeff':<10} {'Accuracy':>10} {'vs Baseline':>12} {'vs Combined':>12}")
print("-"*44)
for c, v in sorted(grid_results.items()):
    print(f"  {c:<8} {v['accuracy']:>10.1f}% {v['accuracy']-51.7:>+11.1f}pp {v['accuracy']-61.7:>+11.1f}pp")
print(f"  {'Baseline':<8} {'51.7':>10}%")
print(f"  {'Combined':<8} {'61.7':>10}%")

best_coeff = max(grid_results, key=lambda c: grid_results[c]["accuracy"])
print(f"\nBest coeff: {best_coeff} ({grid_results[best_coeff]['accuracy']:.1f}%)")



EGDR coeff=0.05 (coeff=0.05, hint=False)


  0%|          | 0/60 [00:00<?, ?it/s]

  Acc=58.3% | EGDR worked=60/60
  Var(C)=0.1450 Var(W)=0.1432 LP gap=+0.0018
  Cav(C)=0.3418 Cav(W)=0.3678 Cav gap=-0.0259

EGDR coeff=0.1 (coeff=0.1, hint=False)


  0%|          | 0/60 [00:00<?, ?it/s]

  Acc=55.0% | EGDR worked=60/60
  Var(C)=0.1455 Var(W)=0.1419 LP gap=+0.0036
  Cav(C)=0.3401 Cav(W)=0.3702 Cav gap=-0.0301

EGDR coeff=0.15 (coeff=0.15, hint=False)


...

  Acc=56.7% | EGDR worked=60/60
  Var(C)=0.1468 Var(W)=0.1423 LP gap=+0.0045
  Cav(C)=0.3391 Cav(W)=0.3718 Cav gap=-0.0327

EGDR coeff=0.2 (coeff=0.2, hint=False)


  0%|          | 0/60 [00:00<?, ?it/s]

  Acc=58.3% | EGDR worked=60/60
  Var(C)=0.1460 Var(W)=0.1421 LP gap=+0.0039
  Cav(C)=0.3409 Cav(W)=0.3699 Cav gap=-0.0290

EGDR coeff=0.3 (coeff=0.3, hint=False)


  0%|          | 0/60 [00:00<?, ?it/s]

  Acc=61.7% | EGDR worked=60/60
  Var(C)=0.1440 Var(W)=0.1457 LP gap=-0.0016
  Cav(C)=0.3440 Cav(W)=0.3670 Cav gap=-0.0230

GRID SEARCH SUMMARY:
Coeff        Accuracy  vs Baseline  vs Combined
--------------------------------------------
  0.05           58.3%        +6.6pp        -3.4pp
  0.1            55.0%        +3.3pp        -6.7pp
  0.15           56.7%        +5.0pp        -5.0pp
  0.2            58.3%        +6.6pp        -3.4pp
  0.3            61.7%       +10.0pp        -0.0pp
  Baseline       51.7%
  Combined       61.7%

Best coeff: 0.3 (61.7%)


In [ ]:
# ── CONDITION G: Hint + EGDR (best coeff from grid search) ──────────────────
BEST_COEFF = best_coeff
results_G, acc_G = run_egdr_condition(
    BEST_COEFF, use_hint=True,
    label=f"G: Hint + EGDR"
)

print(f"\nKEY COMPARISON:")
print(f"  D: Hint + Cache Steer:  61.7%")
print(f"  G: Hint + EGDR:         {acc_G:.1f}%")
print(f"  F: EGDR only:           {grid_results[BEST_COEFF]['accuracy']:.1f}%")
print(f"  A: Baseline:            51.7%")
print(f"  G vs D: {acc_G-61.7:+.1f}pp")



G: Hint + EGDR (coeff=0.3, hint=True)


  0%|          | 0/60 [00:00<?, ?it/s]

  Acc=63.3% | EGDR worked=60/60
  Var(C)=0.1358 Var(W)=0.1392 LP gap=-0.0034
  Cav(C)=0.3389 Cav(W)=0.3456 Cav gap=-0.0067

KEY COMPARISON:
  D: Hint + Cache Steer:  61.7%
  G: Hint + EGDR:         63.3%
  F: EGDR only:           61.7%
  A: Baseline:            51.7%
  G vs D: +1.6pp


In [ ]:
# ── FINAL SUMMARY: All conditions A-D, F, G ──────────────────────────────────
from scipy.stats import ttest_ind
import numpy as np

all_final = {
    "A: Baseline":    results_A,
    "B: Hint":        results_B,
    "C: Cache Steer": results_C,
    "D: Combined":    results_D,
    "F: EGDR":        grid_results[best_coeff]["results"],
    "G: Hint+EGDR":   results_G,
}

print("="*80)
print("COMPLETE RESULTS: Conditions A-D, F (EGDR), G (Hint+EGDR)")
print("="*80)
print(f"{'Condition':<22} {'Acc':>7} {'Var(C)':>8} {'Var(W)':>8} {'LP gap':>8} {'Cav(C)':>8} {'Cav(W)':>8} {'Cav gap':>8}")
print("-"*80)

for cname, res in all_final.items():
    acc = sum(r["correct"] for r in res)/len(res)*100
    vc  = [r["variance"]     for r in res if r.get("correct")     and r.get("variance") is not None]
    vw  = [r["variance"]     for r in res if not r.get("correct") and r.get("variance") is not None]
    cc  = [r["mean_caveman"] for r in res if r.get("correct")     and r.get("mean_caveman") is not None]
    cw  = [r["mean_caveman"] for r in res if not r.get("correct") and r.get("mean_caveman") is not None]
    mvc = float(np.mean(vc)) if vc else float("nan")
    mvw = float(np.mean(vw)) if vw else float("nan")
    mcc = float(np.mean(cc)) if cc else float("nan")
    mcw = float(np.mean(cw)) if cw else float("nan")
    print(f"{cname:<22} {acc:>7.1f}% {mvc:>8.4f} {mvw:>8.4f} {mvc-mvw:>+8.4f} {mcc:>8.4f} {mcw:>8.4f} {mcc-mcw:>+8.4f}")

print()
print("Statistical tests (entropy signal for EGDR conditions):")
from scipy.stats import ttest_ind
for cname in ["F: EGDR", "G: Hint+EGDR"]:
    res = all_final[cname]
    ec  = [r["mean_entropy"] for r in res if r.get("correct")     and r.get("mean_entropy") is not None]
    ew  = [r["mean_entropy"] for r in res if not r.get("correct") and r.get("mean_entropy") is not None]
    if ec and ew:
        _, p = ttest_ind(ec, ew)
        print(f"  {cname}: entropy gap={np.mean(ec)-np.mean(ew):+.4f}, p={p:.3f} {'**' if p<0.01 else '*' if p<0.05 else 'n.s.'}")

print()
print("Grid search:")
for c in sorted(grid_results):
    print(f"  coeff={c}: {grid_results[c]['accuracy']:.1f}% ({grid_results[c]['accuracy']-51.7:+.1f}pp vs baseline)")


COMPLETE RESULTS: Conditions A-D, F (EGDR), G (Hint+EGDR)
Condition                  Acc   Var(C)   Var(W)   LP gap   Cav(C)   Cav(W)  Cav gap
--------------------------------------------------------------------------------
A: Baseline               51.7%   0.1421   0.1402  +0.0018      nan      nan     +nan
B: Hint                   46.7%   0.1467   0.1302  +0.0165      nan      nan     +nan
C: Cache Steer            53.3%   0.1508   0.1470  +0.0038      nan      nan     +nan
D: Combined               65.0%   0.1411   0.1314  +0.0097      nan      nan     +nan
F: EGDR                   61.7%   0.1440   0.1457  -0.0016   0.3440   0.3670  -0.0230
G: Hint+EGDR              63.3%   0.1358   0.1392  -0.0034   0.3389   0.3456  -0.0067

Statistical tests (entropy signal for EGDR conditions):
  F: EGDR: entropy gap=-0.0448, p=0.019 *
  G: Hint+EGDR: entropy gap=-0.0638, p=0.021 *

Grid search:
  coeff=0.05: 58.3% (+6.6pp vs baseline)
  coeff=0.1: 55.0% (+3.3pp vs baseline)
  coeff=0.15: 56.7%

In [ ]:
# ── SAVE RESULTS ─────────────────────────────────────────────────────────────
import json, numpy as np

def safe(v):
    if v is None: return None
    try:
        if hasattr(v,"item"): return v.item()
        return float(v) if isinstance(v,(float,int,np.floating,np.integer)) else str(v)
    except: return None

output = {
    "grid_search": {
        str(c): {
            "accuracy": v["accuracy"],
            "results": [{"idx":r["idx"],"correct":r["correct"],
                         "variance":safe(r.get("variance")),
                         "mean_caveman":safe(r.get("mean_caveman")),
                         "mean_entropy":safe(r.get("mean_entropy"))}
                        for r in v["results"]]
        } for c, v in grid_results.items()
    },
    "condition_G": {
        "coeff": best_coeff,
        "accuracy": sum(r["correct"] for r in results_G)/len(results_G),
        "results": [{"idx":r["idx"],"correct":r["correct"],
                     "variance":safe(r.get("variance")),
                     "mean_caveman":safe(r.get("mean_caveman")),
                     "mean_entropy":safe(r.get("mean_entropy"))}
                    for r in results_G]
    },
    "conditions_AD": {
        cname: [{"idx":r["idx"],"correct":r["correct"],
                 "variance":safe(r.get("variance")),
                 "mean_caveman":safe(r.get("mean_caveman"))}
                for r in res]
        for cname, res in {"A":results_A,"B":results_B,"C":results_C,"D":results_D}.items()
    }
}

with open("egdr_gridsearch_condG.json","w") as f:
    json.dump(output, f, indent=2)
print("Saved: egdr_gridsearch_condG.json")

try:
    from google.colab import files
    files.download("egdr_gridsearch_condG.json")
except:
    print("File saved locally.")
print("\nDone. Paste the FINAL SUMMARY table here.")


Saved: egdr_gridsearch_condG.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Done. Paste the FINAL SUMMARY table here.


In [ ]:
print("Model device map:", model.hf_device_map if hasattr(model, 'hf_device_map') else "single device")
print("Model main device:", next(model.parameters()).device)
print("DEVICE variable:", DEVICE)

Model device map: single device
Model main device: cuda:0
DEVICE variable: cuda


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CONDITION E: Cache Steering + EGDR  (NO HINT, fully answer-free)
# The deployable method: cache-steer sets reasoning mode once at the prompt,
# EGDR applies entropy-gated density correction per token. No gold answer used.
# ════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def generate_cache_egdr(prompt_text, coeff=0.3, cv=1.0, ck=0.0):
    enc        = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=512)
    input_ids  = enc.input_ids.to(DEVICE)
    target_pos = input_ids.shape[1] - 1

    # (1) prompt pass + one-shot cache steering at last prompt token (raw vectors)
    out      = model(input_ids, use_cache=True, return_dict=True)
    kv_cache = out.past_key_values
    for li, layer in enumerate(kv_cache.layers):
        if li < len(steering_vectors["k"]):
            layer.keys[0, :, target_pos, :]   += ck * steering_vectors["k"][li].to(layer.keys.device)
            layer.values[0, :, target_pos, :] += cv * steering_vectors["v"][li].to(layer.values.device)

    generated, lp_log, ent_log = [], [], []
    logits = out.logits[0, -1, :]
    first  = True

    for _ in range(MAX_EGDR_TOKENS):
        if not first:
            out      = model(torch.tensor([[generated[-1]]], device=DEVICE),
                             past_key_values=kv_cache, use_cache=True, return_dict=True)
            logits   = out.logits[0, -1, :]
            kv_cache = out.past_key_values
        first = False

        log_probs = torch.log_softmax(logits.float(), dim=-1)
        probs     = torch.exp(log_probs)
        entropy   = float(-(probs*log_probs).sum().item())
        ent_log.append(entropy)

        tok_id = int(logits.argmax().item())
        lp_log.append(-float(log_probs[tok_id].item()))
        generated.append(tok_id)

        # (2) EGDR entropy gate -> per-token density correction (normalised direction)
        direction = +coeff if entropy > H_HIGH else (-coeff*0.3 if entropy < H_LOW else 0.0)
        if direction != 0.0:
            for li in range(min(len(kv_cache.layers), N_LAYERS)):
                kv_cache.layers[li].keys[:, :, -1, :]   += direction * density_direction_k[li]
                kv_cache.layers[li].values[:, :, -1, :] += direction * density_direction_v[li]

        if tok_id == tokenizer.eos_token_id: break
        if len(generated) > 10 and "####" in tokenizer.decode(generated, skip_special_tokens=True): break

    return tokenizer.decode(generated, skip_special_tokens=True), lp_log, ent_log

print("generate_cache_egdr() ready  ->  Condition E (Cache+EGDR, answer-free)")


In [ ]:
# ── Run Condition E (Cache+EGDR, no hint) at best grid coeff ─────────────────
from scipy.stats import ttest_ind
import numpy as np

E_COEFF = best_coeff if "best_coeff" in globals() else 0.3

def run_cache_egdr(coeff=E_COEFF):
    results = []
    for i, prob in enumerate(tqdm(eval_problems)):
        q, ans = prob["question"], prob["answer"]
        try:
            response, lp_log, ent_log = generate_cache_egdr(make_prompt(q), coeff=coeff)
            ok = True
        except Exception as ex:
            print(f"  Problem {i}: {ex} -- greedy fallback")
            response, lp_log, ent_log, ok = generate_greedy(make_prompt(q)), [], [], False
        correct = is_correct(response, ans)
        steps   = parse_steps(response)
        if lp_log and len(steps) >= 2 and ok:
            tps      = max(1, len(lp_log)//len(steps))
            step_lps = [float(np.mean(lp_log[s*tps:(s+1)*tps if s<len(steps)-1 else len(lp_log)]))
                        for s in range(len(steps))]
            arr = np.array(step_lps); rng = arr.max()-arr.min()
            variance = float(np.var((arr-arr.min())/rng)) if rng > 1e-9 else 0.0
        else:
            variance = None
        cav = [caveman_ratio(s) for s in steps]
        results.append({"idx":i,"correct":correct,"variance":variance,
                        "mean_caveman":float(np.mean(cav)) if cav else None,
                        "mean_entropy":float(np.mean(ent_log)) if ent_log else None,
                        "egdr_ok":ok})
    return results

print(f"\n{'='*55}\nE: Cache Steer + EGDR (coeff={E_COEFF}, NO hint)\n{'='*55}")
results_E = run_cache_egdr(E_COEFF)

accE = sum(r["correct"] for r in results_E)/len(results_E)*100
ec = [r["mean_entropy"] for r in results_E if r["correct"]     and r["mean_entropy"] is not None]
ew = [r["mean_entropy"] for r in results_E if not r["correct"] and r["mean_entropy"] is not None]
cc = [r["mean_caveman"] for r in results_E if r["correct"]     and r["mean_caveman"] is not None]
cw = [r["mean_caveman"] for r in results_E if not r["correct"] and r["mean_caveman"] is not None]
pE = ttest_ind(ec, ew)[1] if ec and ew else float("nan")
print(f"  Acc={accE:.1f}%")
print(f"  Entropy gap={np.mean(ec)-np.mean(ew):+.4f}, p={pE:.3f} {'**' if pE<0.01 else '*' if pE<0.05 else 'n.s.'}")
print(f"  Caveman gap={np.mean(cc)-np.mean(cw):+.4f}")
print(f"\n  vs D (Hint+Cache, oracle): {accE - 65.0:+.1f}pp   [E is answer-free]")


In [ ]:
# ── Blind-hint ablation: hints from the QUESTION ONLY (no gold answer) ───────
# Shows the hint conditions (B, D) collapse without oracle access to the answer,
# which is exactly why an answer-free method (E) is the deployable solution.
BLIND_SYS = ("You are a math tutor. Given ONLY a question, write a concise high-level "
             "hint (1-2 sentences) on how to approach it. Do NOT solve it or give numbers.")
def generate_blind_hint(question):
    p = tokenizer.apply_chat_template(
        [{"role":"system","content":BLIND_SYS},
         {"role":"user","content":f"Question: {question}\n\nProvide an abstract hint:"}],
        tokenize=False, add_generation_prompt=True)
    return generate_greedy(p, max_new_tokens=80).strip()

print("Generating BLIND hints (no answer)...")
blind_hints = [generate_blind_hint(p["question"]) for p in tqdm(eval_problems)]

def run_with_hintlist(prompt_fn, hint_list):
    res = []
    for i, prob in enumerate(tqdm(eval_problems)):
        q, ans, h = prob["question"], prob["answer"], hint_list[i]
        try:    response = prompt_fn(q, h)
        except Exception: response = ""
        res.append({"correct": is_correct(response, ans)})
    return sum(r["correct"] for r in res)/len(res)*100

accB_blind = run_with_hintlist(lambda q,h: generate_greedy(make_prompt(q, hint=h)), blind_hints)
accD_blind = run_with_hintlist(lambda q,h: generate_with_steered_cache(make_prompt(q, hint=h),
                                            steering_vectors, ck=CK, cv=CV), blind_hints)

print("\n── Hint availability ablation ──")
print(f"  B (Hint)        oracle=46.7%   blind={accB_blind:.1f}%")
print(f"  D (Hint+Cache)  oracle=65.0%   blind={accD_blind:.1f}%")
print(f"  E (Cache+EGDR)  answer-free={accE:.1f}%  <- needs no hint at all")


In [ ]:
# ── FINAL TABLE incl. E + answer-free ranking ───────────────────────────────
print("="*72)
print(f"{'Condition':<26}{'Acc':>7}{'Answer-free?':>14}")
print("-"*72)
rows = [("A: Baseline",51.7,"yes"),("B: Hint",46.7,"NO (oracle)"),
        ("C: Cache Steer",53.3,"yes"),("D: Hint+Cache",65.0,"NO (oracle)"),
        ("F: EGDR",61.7,"yes"),("G: Hint+EGDR",63.3,"NO (oracle)"),
        (f"E: Cache+EGDR",round(accE,1),"YES")]
for n,a,af in rows:
    print(f"{n:<26}{a:>6.1f}%{af:>14}")
print("-"*72)
print("Best ANSWER-FREE method = E (Cache+EGDR). This is the deployable result.")
print()
print("# ── To get the 'any model' claim, re-run the whole notebook with:")
print("#     MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'   (done)")
print("#     MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'")
print("#     MODEL_NAME = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'")
print("#   All three fit on a single T4. Report E across all three.")
